## Learning notes — 04. Process Orders Data

### Learning objectives
Follow **04. Process Orders Data** from its source inputs through its operations to the archived results.
- Explain stream, autoloader, ctas, json-array, profiling as used in this lesson.

### Source-specific learning focus
- Follow incremental ingestion from discovery to persisted results.
- Separate file discovery, parsing and schema tracking.
- Distinguish query results from materialized table state.
- Follow nested JSON and arrays as they become columns or rows.
- Interpret selected fields, samples and counts at their actual output grain.

### Input/output interpretation
Original source and outputs are preserved unchanged and were not executed. Saved tables, HTML displays and errors are archived generated outputs, not supplied datasets. Inline literals are examples already present in the source. Cloud storage, remote tables, catalogs, services and checkpoints are unavailable locally unless a supplied snapshot is explicitly listed below; no external input is claimed available.


## Learning notes — supplied input snapshot

Path: `data/circuitbox/orders/orders_day1.json`

SHA-256: `e653657907fe6cbaeeec87124fb53c86a955f26add32da619eac1c5d5f5e9904`

Complete supplied text (not generated output):

```json
{"order_id": 1, "customer_id": 1001, "order_timestamp": "2024-10-28 04:47:27", "payment_method": "Credit Card", "items": [{"item_id": 10, "name": "Drone", "category": "Electronics", "price": 799, "quantity": 1}], "order_status": "Pending"}
{"order_id": 2, "customer_id": 1001, "order_timestamp": "2024-10-07 22:09:27", "payment_method": "Credit Card", "items": [{"item_id": 5, "name": "Tablet", "category": "Electronics", "price": 399, "quantity": 1}], "order_status": "Pending"}
{"order_id": 3, "customer_id": 1002, "order_timestamp": "2024-10-13 17:34:19", "payment_method": "Bank Transfer", "items": [{"item_id": 4, "name": "Smartwatch", "category": "Electronics", "price": 299, "quantity": 3}], "order_status": "Unknown"}
{"order_id": 4, "customer_id": 1003, "order_timestamp": "2024-10-20 01:47:25", "payment_method": "Bank Transfer", "items": [{"item_id": 9, "name": "Smart TV", "category": "Electronics", "price": 1199, "quantity": 1}], "order_status": "Completed"}
{"order_id": 5, "customer_id": 1004, "order_timestamp": "2024-10-13 12:40:26", "payment_method": "PayPal", "items": [{"item_id": 8, "name": "Gaming Console", "category": "Electronics", "price": 499, "quantity": 1}], "order_status": "Shipped"}
{"order_id": 6, "customer_id": 1005, "order_timestamp": "2024-10-29 21:39:19", "payment_method": "PayPal", "items": [{"item_id": 7, "name": "External Hard Drive", "category": "Electronics", "price": 129, "quantity": 3}], "order_status": "Unknown"}
{"order_id": 7, "customer_id": 1006, "order_timestamp": "2024-10-25 05:49:16", "payment_method": "Bank Transfer", "items": [{"item_id": 1, "name": "Smartphone", "category": "Electronics", "price": 699, "quantity": 1}], "order_status": "Shipped"}
{"order_id": 8, "customer_id": 1006, "order_timestamp": "2024-10-23 02:51:05", "payment_method": "PayPal", "items": [{"item_id": 8, "name": "Gaming Console", "category": "Electronics", "price": 499, "quantity": 1}], "order_status": "Pending"}
{"order_id": 9, "customer_id": 1007, "order_timestamp": "2024-10-06 10:23:28", "payment_method": "Unknown", "items": [{"item_id": 3, "name": "Wireless Headphones", "category": "Electronics", "price": 199, "quantity": 1}, {"item_id": 7, "name": "External Hard Drive", "category": "Electronics", "price": 129, "quantity": 1}], "order_status": "Cancelled"}
{"order_id": 10, "customer_id": 1008, "order_timestamp": "2024-10-15 13:09:33", "payment_method": "Credit Card", "items": [{"item_id": 3, "name": "Wireless Headphones", "category": "Electronics", "price": 199, "quantity": 2}], "order_status": "Pending"}
{"order_id": 11, "customer_id": 1009, "order_timestamp": "2024-10-25 03:37:42", "payment_method": "PayPal", "items": [{"item_id": 10, "name": "Drone", "category": "Electronics", "price": 799, "quantity": 1}, {"item_id": 3, "name": "Wireless Headphones", "category": "Electronics", "price": 199, "quantity": 1}], "order_status": "Completed"}
{"order_id": 12, "customer_id": 1010, "order_timestamp": "2024-10-23 20:06:42", "payment_method": "Bank Transfer", "items": [{"item_id": 8, "name": "Gaming Console", "category": "Electronics", "price": 499, "quantity": 2}], "order_status": "Shipped"}
```

## Learning notes — supplied input snapshot

Path: `data/circuitbox/orders/orders_day2.json`

SHA-256: `814e6982853274d17de0e58559268b915725e6803d70c99011fff199a852fadf`

Complete supplied text (not generated output):

```json
{"order_id": 13, "customer_id": 1011, "order_timestamp": "2024-11-16 23:07:31", "payment_method": "Bank Transfer", "items": [{"item_id": 8, "name": "Gaming Console", "category": "Electronics", "price": 499, "quantity": 2}], "order_status": "Cancelled"}
{"order_id": 14, "customer_id": 1012, "order_timestamp": "2024-11-19 19:18:48", "payment_method": "Credit Card", "items": [{"item_id": 7, "name": "External Hard Drive", "category": "Electronics", "price": 129, "quantity": 2}], "order_status": "Pending"}
{"order_id": 15, "customer_id": 1013, "order_timestamp": "2024-11-27 13:15:29", "payment_method": "Bank Transfer", "items": [{"item_id": 2, "name": "Laptop", "category": "Electronics", "price": 999, "quantity": 1}], "order_status": "Shipped"}
{"order_id": 16, "customer_id": 1013, "order_timestamp": "2024-11-18 02:38:21", "payment_method": "PayPal", "items": [{"item_id": 5, "name": "Tablet", "category": "Electronics", "price": 399, "quantity": 2}], "order_status": "Pending"}
```

# Archived Databricks example

Historical source and outputs, not locally executable. Nothing was run during conversion. Databricks services, datasets and original image assets may be unavailable; image references are preserved, not downloaded. Outputs and errors are archived, not new execution.

## Process Orders Data 
1. Ingest the data into the data lakehouse - bronze_orders
1. Perform data quality checks and transform the data as required - silver_orders_clean
1. Explode the items array from the order object - silver_orders

![dea06-04-process-orders-data-1.png](images/dea06-04-process-orders-data-1.png)

### 1. Ingest the data into the data lakehouse - bronze_orders

## Learning notes — before source operation

**stream**: A streaming input is processed incrementally, not as a fixed batch. Interpret displayed rows as historical observations, not proof that a stream is currently running.

**autoloader**: Auto Loader discovers arriving files. File format/options and schema tracking determine parsing; discovery is distinct from downstream transformations.

**ctas**: Table creation materializes a query/DataFrame into a destination. Distinguish its stored schema and rows from a view and check whether replacement overwrites existing state.

**profiling**: Read the selected columns and output grain before interpreting the saved result. Counts, nulls and samples describe the archived input state, not a newly executed validation.

Archived Databricks source — shown for reading, not Python execution.

```sql
CREATE OR REFRESH STREAMING TABLE bronze_orders
COMMENT "Raw orders data ingested from the source system"
TBLPROPERTIES ("quality" = "bronze")
AS
SELECT *,
        _metadata.file_path AS input_file_path,
       CURRENT_TIMESTAMP AS ingestion_timestamp
  FROM cloud_files(
                    '/Volumes/circuitbox/landing/operational_data/orders/', 
                    'json',
                    map("cloudFiles.inferColumnTypes", "true")
                   );
```

### 2. Perform data quality checks and transform the data as required - silver_orders_clean

## Learning notes — before source operation

**stream**: A streaming input is processed incrementally, not as a fixed batch. Interpret displayed rows as historical observations, not proof that a stream is currently running.

**ctas**: Table creation materializes a query/DataFrame into a destination. Distinguish its stored schema and rows from a view and check whether replacement overwrites existing state.

**profiling**: Read the selected columns and output grain before interpreting the saved result. Counts, nulls and samples describe the archived input state, not a newly executed validation.

Archived Databricks source — shown for reading, not Python execution.

```sql
CREATE OR REFRESH STREAMING TABLE silver_orders_clean(
  CONSTRAINT valid_customer_id EXPECT (customer_id IS NOT NULL) ON VIOLATION FAIL UPDATE,
  CONSTRAINT valid_order_id EXPECT (order_id IS NOT NULL) ON VIOLATION FAIL UPDATE,
  CONSTRAINT valid_order_status EXPECT (order_status IN ('Pending', 'Shipped', 'Cancelled', 'Completed')),
  CONSTRAINT valid_payment_method EXPECT (payment_method IN ('Credit Card', 'Bank Transfer', 'PayPal'))
)
COMMENT "Cleaned orders data"
TBLPROPERTIES ("quality" = "silver")
AS
SELECT order_id,
       customer_id,
       CAST(order_timestamp AS TIMESTAMP) AS order_timestamp,
       payment_method,
       items,
       order_status
  FROM STREAM(LIVE.bronze_orders);
```

### 3. Explode the items array from the order object - silver_orders

## Learning notes — before source operation

**stream**: A streaming input is processed incrementally, not as a fixed batch. Interpret displayed rows as historical observations, not proof that a stream is currently running.

**ctas**: Table creation materializes a query/DataFrame into a destination. Distinguish its stored schema and rows from a view and check whether replacement overwrites existing state.

**json-array**: JSON extraction and array operations change structure: parsing creates typed values, while explode can create multiple rows per input. Inspect missing fields, null arrays and empty arrays.

**Unexecuted teaching example (illustrative, not an archived or generated result):** Exploding an array [a, b] yields two rows for that input row.

**profiling**: Read the selected columns and output grain before interpreting the saved result. Counts, nulls and samples describe the archived input state, not a newly executed validation.

Archived Databricks source — shown for reading, not Python execution.

```sql
CREATE STREAMING TABLE silver_orders
AS
SELECT order_id,
      customer_id,
      order_timestamp,
      payment_method,
      order_status,
      item.item_id,
      item.name AS item_name,
      item.price AS item_price,
      item.quantity AS item_quantity,
      item.category AS item_category
  FROM (SELECT order_id,
              customer_id,
              order_timestamp,
              payment_method,
              order_status,
              explode(items) AS item
          FROM STREAM(LIVE.silver_orders_clean));
```